# Séance 10 · Parler à un LLM par le code · ⭐⭐

**Niveau : ⭐⭐ Intermédiaire**

La semaine dernière, on a compris comment un LLM fonctionne. Aujourd'hui on lui **parle depuis Python** : on lui envoie des messages, on lui donne une personnalité et des règles, on lui fait garder la mémoire d'une conversation, et on lui demande des réponses en JSON pour les réutiliser dans un programme.

Tout tourne dans **Google Colab** (menu *Exécution → Modifier le type d'exécution → T4 GPU*). Exécute chaque cellule avec `Maj + Entrée`.

**Livrable de la séance** : un chatbot à thème (coach de révisions, générateur de quiz ou maître de jeu de rôle) qui fonctionne, avec sa personnalité écrite par toi.


## Préparation

La même cellule qu'à la séance 9 : elle prépare `llm(messages)`. Tout le notebook n'utilise que cette fonction.

In [ ]:
USE_MODEL = True   # ← mets False pour tester le notebook sans modèle (réponses factices, sans GPU)

import json, re

# ---------- Mode démo : un faux LLM qui répond sans réseau ni GPU ----------
def llm_factice(messages):
    """Réponses écrites à la main, choisies selon les mots du prompt (mode démo)."""
    q = messages[-1]["content"]
    ql = q.lower()
    systeme = " ".join(m["content"] for m in messages if m["role"] == "system").lower()
    tout = " ".join(m["content"] for m in messages).lower()
    if "json" in ql or "json" in systeme:                 # section 5 : sortie structurée
        if "quiz" in tout or "question" in tout:
            return ('Voici le quiz : {"question": "Quel est le type de Pikachu ?", '
                    '"choix": ["Feu", "Électrique", "Eau"], "bonne_reponse": "Électrique"}')
        if "pok" in tout:
            return '{"nom": "Pikachu", "type": "Électrique", "pv": 35, "attaque_preferee": "Éclair"}'
        return '{"reponse": "Paris", "confiance": 0.9}'
    prenom = re.search(r"je m'appelle (\w+)", tout)
    if "prénom" in ql or "souviens" in ql:                # section 4 : l'historique
        return f"Bien sûr, tu t'appelles {prenom.group(1).capitalize()} !" if prenom else "Tu ne me l'as pas encore dit !"
    if "pirate" in systeme:
        return "Arrr ! Moussaillon, hisse tes cahiers et cap sur les fractions, le trésor est au bout !"
    if "maître du jeu" in systeme or "mj" in systeme:
        return "Tu entres dans la taverne. Un nain te fixe et pose une carte sur la table. Que fais-tu ? (1) lui parler (2) prendre la carte"
    if "quiz" in systeme:
        return "Question 1 : quel est le type de Salamèche ? A) Eau B) Feu C) Plante"
    if "coach" in systeme:
        return "Super, on y va ! Commence par 25 minutes de révision, puis 5 minutes de pause. Sur quelle matière on attaque ?"
    if "capitale" in ql:
        return "La capitale de la France est Paris."
    if "bonjour" in ql or "salut" in ql:
        return "Salut ! Je suis là pour t'aider. Qu'est-ce qu'on fait aujourd'hui ?"
    return "Bonne question ! En résumé : c'est un sujet intéressant, et je peux t'en dire plus si tu veux."

# ---------- Le vrai modèle : petit modèle ouvert, gratuit, sans clé ----------

if USE_MODEL:
    %pip install -q transformers accelerate
    from transformers import pipeline
    _pipe = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")

def llm(messages, max_new_tokens=150, temperature=0.7):
    """Envoie une liste de messages au modèle et renvoie sa réponse (du texte)."""
    if not USE_MODEL:
        return llm_factice(messages)
    if temperature == 0:      # température 0 = toujours la réponse la plus probable
        sortie = _pipe(messages, max_new_tokens=max_new_tokens, do_sample=False)
    else:
        sortie = _pipe(messages, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature)
    return sortie[0]["generated_text"][-1]["content"].strip()

print("Modèle prêt :", "Qwen2.5-0.5B-Instruct" if USE_MODEL else "mode démo (llm_factice)")

**En option : le même appel avec une API.** Sur Colab, la clé est fournie par le formateur et rangée dans les **Secrets** (icône 🔑 à gauche), jamais dans le code. Décommente la cellule ci-dessous pour remplacer `llm()` par un gros modèle dans le cloud. Tout le reste du notebook ne change pas : il n'appelle que `llm(messages)`.

In [ ]:
# --- Variante API (à décommenter si le formateur a donné une clé) ---
# %pip install -q anthropic
# from google.colab import userdata          # les Secrets de Colab
# import anthropic
# client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
#
# def llm(messages, max_new_tokens=150, temperature=0.7):
#     systeme = " ".join(m["content"] for m in messages if m["role"] == "system")
#     autres = [m for m in messages if m["role"] != "system"]
#     rep = client.messages.create(model="claude-opus-5", max_tokens=max_new_tokens,
#                                  system=systeme or "Tu es un assistant sympa.", messages=autres)
#     return rep.content[0].text.strip()
#
# Même idée avec Mistral : pip install mistralai, clé dans userdata.get("MISTRAL_API_KEY"),
# client.chat.complete(model="mistral-small-latest", messages=messages).choices[0].message.content

## 1. C'est quoi une API ?

Une **API** (*Application Programming Interface*), c'est un serveur de restaurant. Tu ne rentres pas en cuisine : tu passes une commande précise (la **requête**), le serveur l'apporte en cuisine, et revient avec le plat (la **réponse**). La cuisine peut être à Paris ou à l'autre bout du monde, peu importe.

Sur internet, la commande est une URL et la réponse est souvent du **JSON**, un format texte que Python lit comme un dictionnaire. On l'a déjà fait à la séance 4 avec la PokéAPI : regarde comme ça ressemble à ce qu'on va faire avec un LLM.

In [ ]:
import requests

try:
    reponse = requests.get("https://pokeapi.co/api/v2/pokemon/pikachu", timeout=10)
    pika = reponse.json()                     # la réponse JSON devient un dictionnaire Python
except Exception as e:
    print("Pas de réseau ?", e, "→ on utilise des données de secours")
    pika = {"name": "pikachu", "height": 4, "weight": 60, "types": [{"type": {"name": "electric"}}]}

print("Commande : GET /pokemon/pikachu")
print("Plat reçu :", pika["name"], "| taille", pika["height"], "| poids", pika["weight"], "| type", pika["types"][0]["type"]["name"])

Avec un LLM c'est pareil : on envoie une liste de messages, on reçoit du texte. Que le modèle tourne dans Colab (notre cas) ou chez Anthropic, OpenAI ou Mistral (avec une clé), le code qui suit est **le même** : `llm(messages)`.

## 2. Appeler `llm(messages)` depuis Python

La commande qu'on passe au modèle, c'est une **liste de messages**. Chaque message est un dictionnaire avec un **rôle** (`role`) et un **contenu** (`content`). Pour l'instant, un seul message de rôle `user` : toi.

In [ ]:
messages = [
    {"role": "user", "content": "Quelle est la capitale de la France ? Réponds en une phrase."}
]
reponse = llm(messages)
print(reponse)

**Exercice** : écris une fonction `poser(question)` qui construit la liste de messages et renvoie la réponse, puis pose 3 questions dans une boucle `for`.

In [ ]:
# À toi
def poser(question):
    messages = [{"role": "user", "content": question}]
    return llm(messages)

for q in ["Bonjour, qui es-tu ?", "Quelle est la capitale de la France ?"]:
    print("Q :", q)
    print("R :", poser(q), "\n")

<details><summary>Solution</summary>

```python
def poser(question):
    return llm([{"role": "user", "content": question}])

questions = ["Bonjour, qui es-tu ?", "Quelle est la capitale de la France ?", "Donne-moi une idée de projet Python."]
for q in questions:
    print("Q :", q)
    print("R :", poser(q), "\n")
```

</details>

## 3. Les rôles : system, user, assistant

Il y a trois rôles dans une conversation :
- **`system`** : les consignes données au modèle avant tout (sa **personnalité** et ses **règles**). L'utilisateur ne le voit pas.
- **`user`** : ce que dit l'utilisateur.
- **`assistant`** : ce que le modèle a répondu.

Le **prompt système**, c'est la fiche de poste de ton assistant : qui il est, comment il parle, ce qu'il a le droit de faire ou pas. C'est la partie la plus importante de ton futur chatbot.

In [ ]:
def demander(question, systeme="Tu es un assistant sympa qui répond en français, en 3 phrases maximum."):
    messages = [
        {"role": "system", "content": systeme},
        {"role": "user", "content": question},
    ]
    return llm(messages)

print(demander("Donne-moi un conseil pour réviser mes maths."))
print()
print(demander("Donne-moi un conseil pour réviser mes maths.",
               systeme="Tu es un pirate. Tu parles comme un pirate, en français, avec des « Arrr »."))

Un bon prompt système contient : **qui** (rôle), **comment** (ton, langue, longueur), **quoi faire** (la tâche), **quoi ne pas faire** (les règles). Exemple :

> Tu es un coach de révisions pour un débutant. Tu tutoies, tu réponds en français en 3 phrases maximum. Tu poses toujours une question à la fin pour faire avancer. Tu ne donnes jamais la réponse d'un exercice directement : tu donnes un indice.

**Exercice** : écris un prompt système avec ces 4 éléments pour un assistant de ton choix, et teste-le avec 2 questions différentes.

In [ ]:
# À toi
mon_systeme = """Tu es un coach de révisions pour un débutant. Tu tutoies, tu réponds en français en 3 phrases maximum.
Tu poses toujours une question à la fin. Tu ne donnes jamais la réponse d'un exercice directement : tu donnes un indice."""

print(demander("J'ai un contrôle de maths demain et je stresse.", systeme=mon_systeme))

<details><summary>Solution</summary>

```python
mon_systeme = """Tu es un prof d'histoire passionné. Tu vouvoies, tu réponds en français en 2 phrases maximum,
toujours avec une anecdote. Tu ne parles jamais d'autre chose que d'histoire : si on te demande autre chose, tu ramènes à l'histoire."""
print(demander("Raconte-moi Napoléon.", systeme=mon_systeme))
print(demander("Tu aimes les jeux vidéo ?", systeme=mon_systeme))   # il doit ramener à l'histoire
```

</details>

## 4. L'historique : un chatbot qui se souvient

Jusqu'ici, chaque question repart de zéro : le modèle n'a **aucune mémoire**. Un chatbot, c'est juste un programme qui **garde la liste de tous les messages** et la renvoie entière au modèle à chaque tour. Le modèle « se souvient » parce qu'on lui remontre toute la conversation.

Analogie : parler à quelqu'un qui perd la mémoire toutes les 5 secondes, mais à qui on tend à chaque fois le compte-rendu complet de la discussion.

In [ ]:
class Chatbot:
    """Un chatbot = un prompt système + l'historique des messages."""

    def __init__(self, systeme):
        self.historique = [{"role": "system", "content": systeme}]

    def parler(self, message):
        self.historique.append({"role": "user", "content": message})
        reponse = llm(self.historique)
        self.historique.append({"role": "assistant", "content": reponse})
        return reponse

coach = Chatbot("Tu es un coach sportif très motivé. Tu réponds en français, en 2 phrases maximum.")
print(coach.parler("Salut, je m'appelle Nova et je veux progresser en course à pied."))
print(coach.parler("Tu te souviens de mon prénom ?"))
print("\nNombre de messages dans l'historique :", len(coach.historique))

**Exercice** : tiens une conversation de 5 messages avec ton propre chatbot (personnalité au choix). À quel moment perd-il le fil ? Combien de tokens pèse l'historique à la fin ? (Compte-les avec `tiktoken` comme à la séance 9 : plus l'historique grandit, plus chaque tour coûte cher.)

In [ ]:
# À toi
bot = Chatbot("Tu es un guide touristique de Paris, enthousiaste. Tu réponds en français, en 2 phrases.")
for message in ["Bonjour ! Je m'appelle Lina.", "Que visiter en premier ?", "Tu te souviens de mon prénom ?"]:
    print("Moi :", message)
    print("Bot :", bot.parler(message), "\n")

<details><summary>Solution</summary>

```python
try:
    import tiktoken
except ImportError:
    %pip install -q tiktoken
    import tiktoken
enc = tiktoken.get_encoding("cl100k_base")
texte_total = " ".join(m["content"] for m in bot.historique)
print("Tokens dans l'historique :", len(enc.encode(texte_total)))
# Un petit modèle perd le fil après quelques tours ; un gros tient des dizaines de pages.
# Mais tout l'historique est renvoyé à chaque tour : c'est pour ça que les longues conversations coûtent cher.
```

</details>

## 5. Sortie structurée : demander du JSON

Du texte libre, c'est bien pour lire. Mais pour **réutiliser** la réponse dans un programme (afficher un quiz, remplir un tableau, faire un calcul), il faut un format fixe : le **JSON**. On demande au modèle de répondre *uniquement* en JSON, puis on le transforme en dictionnaire avec `json.loads`.

Deux pièges : le modèle ajoute parfois du texte autour (« Voici le JSON : {...} »), et parfois le JSON est cassé. D'où : on **extrait** la partie entre accolades, on **essaie** de la lire, et si ça rate on **redemande**.

In [ ]:
def extraire_json(texte):
    """Récupère la partie {...} d'un texte et la transforme en dictionnaire. Lève une erreur si impossible."""
    debut, fin = texte.find("{"), texte.rfind("}")
    if debut == -1 or fin == -1:
        raise ValueError("pas d'accolades dans la réponse")
    return json.loads(texte[debut:fin + 1])

def demander_json(question, systeme, essais=3):
    """Demande un JSON au modèle, ré-essaie si la réponse n'est pas lisible."""
    for essai in range(1, essais + 1):
        texte = llm([{"role": "system", "content": systeme}, {"role": "user", "content": question}], temperature=0.3)
        try:
            return extraire_json(texte)
        except (ValueError, json.JSONDecodeError) as e:
            print(f"essai {essai} : JSON illisible ({e}) → on redemande")
    return None

SYSTEME_JSON = "Tu réponds UNIQUEMENT avec un objet JSON valide, sans texte autour, sans explication."
fiche = demander_json("Donne une fiche du Pokémon Pikachu avec les clés : nom, type, pv, attaque_preferee.", SYSTEME_JSON)
print(fiche)
print("Type de la réponse :", type(fiche).__name__, "| type de Pikachu :", fiche["type"] if fiche else "?")

**Exercice** : demande un quiz en JSON avec les clés `question`, `choix` (liste de 3) et `bonne_reponse`, puis affiche-le joliment : la question, les choix numérotés, et la bonne réponse en dernier.

In [ ]:
# À toi
quiz = demander_json("Écris une question de quiz sur les Pokémon, en JSON avec les clés : question, choix (liste de 3), bonne_reponse.", SYSTEME_JSON)
print(quiz)

<details><summary>Solution</summary>

```python
quiz = demander_json("Écris une question de quiz sur les Pokémon, en JSON avec les clés : question, choix (liste de 3), bonne_reponse.", SYSTEME_JSON)
if quiz:
    print("❓", quiz["question"])
    for i, c in enumerate(quiz["choix"], 1):
        print(f"   {i}. {c}")
    print("✅ Bonne réponse :", quiz["bonne_reponse"])
```

</details>

## 6. Projet : ton bot à thème (80 min)

Choisis **un** thème et construis ton chatbot avec la classe `Chatbot` de la section 4. Il doit avoir une vraie personnalité, des règles, et tenir une conversation de plusieurs tours.

| Thème | Ce qu'il fait | Idée de règle |
|---|---|---|
| **Coach de révisions** | pose des questions sur une matière, encourage, donne des indices | ne jamais donner la réponse directement |
| **Générateur de quiz** | pose une question à choix multiples, corrige, compte les points | une seule question à la fois, format fixe |
| **Maître de jeu (MJ)** | raconte une aventure, propose 2 choix à chaque tour | rester dans l'univers choisi, toujours finir par « Que fais-tu ? » |

Étapes : (1) écris le prompt système, (2) teste avec le scénario, (3) ajoute une fonctionnalité JSON (score, inventaire, fiche), (4) si tu as le temps, l'interface Streamlit.

In [ ]:
# Question 1 : ton prompt système (qui / comment / quoi faire / quoi ne pas faire)
PERSONNALITE = """Tu es un maître du jeu de rôle dans un univers médiéval-fantastique.
Tu tutoies le joueur, tu réponds en français, en 3 phrases maximum.
À chaque tour tu décris la scène puis tu proposes exactement 2 choix numérotés (1) et (2).
Tu ne sors jamais de l'univers et tu termines toujours par : Que fais-tu ?"""

mon_bot = Chatbot(PERSONNALITE)

In [ ]:
# Question 2 : le scénario de test (pas de input() : une liste de messages, pour que ça tourne partout)
scenario = [
    "Bonjour, je m'appelle Nova, je suis une archère elfe.",
    "Je choisis le choix 1.",
    "Tu te souviens de mon nom et de ma classe ?",
]
for message in scenario:
    print("Moi :", message)
    print("Bot :", mon_bot.parler(message), "\n")

In [ ]:
# Question 3 : une fonctionnalité structurée en JSON (à adapter à ton thème)
#   coach → {"matiere": ..., "question": ..., "indice": ...}
#   quiz  → {"question": ..., "choix": [...], "bonne_reponse": ...}
#   MJ    → {"lieu": ..., "objets": [...], "pv": ...}
fiche = demander_json("Résume l'état actuel du joueur en JSON avec les clés : nom, classe, lieu, pv.", SYSTEME_JSON)
print(fiche)

In [ ]:
# Question 4 : (facultatif) mode interactif dans Colab, désactivé par défaut pour que le notebook s'exécute d'un coup
MODE_INTERACTIF = False
if MODE_INTERACTIF:
    while True:
        message = input("Moi (ou 'stop') : ")
        if message.lower() == "stop":
            break
        print("Bot :", mon_bot.parler(message))

### Bonus : une vraie interface avec Streamlit

**Streamlit** transforme un script Python en page web en quelques lignes. La cellule ci-dessous n'est pas exécutée par le notebook : elle **écrit** un fichier `app.py` (sur ta machine, lance ensuite `streamlit run app.py`). Colle ta cellule « Préparation » à la place de la fonction `llm` d'exemple.

In [ ]:
%%writefile app.py
import streamlit as st

def llm(messages):                      # ← remplace par la fonction llm de la cellule Préparation
    return "Réponse d'exemple : colle ici ta vraie fonction llm."

PERSONNALITE = "Tu es un coach de révisions sympa. Tu réponds en français, en 3 phrases maximum."

st.title("Mon chatbot")
if "historique" not in st.session_state:                      # la mémoire de la page
    st.session_state.historique = [{"role": "system", "content": PERSONNALITE}]

for m in st.session_state.historique[1:]:                      # on ré-affiche la conversation
    st.chat_message(m["role"]).write(m["content"])

if message := st.chat_input("Écris ton message"):
    st.session_state.historique.append({"role": "user", "content": message})
    st.chat_message("user").write(message)
    reponse = llm(st.session_state.historique)
    st.session_state.historique.append({"role": "assistant", "content": reponse})
    st.chat_message("assistant").write(reponse)

Pour lancer l'interface sur ton ordinateur : `pip install streamlit` puis `streamlit run app.py`. Une page web s'ouvre avec ton chatbot. (Dans Colab, c'est possible mais un peu plus compliqué : demande au formateur.)

## À retenir

- Une **API**, c'est un serveur de restaurant : une commande précise part, une réponse revient. Un LLM s'utilise comme ça depuis Python.
- On lui envoie une **liste de messages**, chacun avec un rôle : `system` (consignes), `user` (toi), `assistant` (lui).
- Le **prompt système** = la fiche de poste : qui, comment, quoi faire, quoi ne pas faire.
- Un modèle n'a **aucune mémoire** : le chatbot renvoie tout l'**historique** à chaque tour. Plus il est long, plus ça coûte.
- Pour réutiliser une réponse dans un programme : demander du **JSON**, l'extraire, `json.loads` dans un `try/except`, redemander si ça rate.
- Que le modèle soit petit (Colab) ou gros (API avec une clé), **le code est le même**. La clé, elle, ne va jamais dans le code.

## Pour montrer aux autres

Fais une démo de ton bot en 2 minutes en répondant à :
1. Quelle personnalité et quelles règles lui as-tu données ? Montre ton prompt système.
2. Une règle qu'il respecte bien, et une qu'il oublie parfois : pourquoi, à ton avis ?
3. Qu'est-ce que tu ajouterais s'il tournait sur un gros modèle avec une clé d'API ?

Liens gratuits
- Streamlit (interface web en Python) : https://streamlit.io
- Documentation de l'API Anthropic : https://docs.anthropic.com
- Documentation de l'API Mistral : https://docs.mistral.ai
- Un terrain de jeu pour tester des prompts : https://huggingface.co/chat